# 362. Design Hit Counter

**Difficulty:** Medium &nbsp;|&nbsp; **Topics:** design, queue, circular-buffer
&nbsp;|&nbsp; [LeetCode](https://leetcode.com/problems/design-hit-counter/)

Design a hit counter which counts the number of hits received in the past 5
minutes (that is, the past `300` seconds).

Your system should accept a `timestamp` parameter (**in seconds**), and you may
assume that calls are being made to the system in chronological order - that is,
`timestamp` is **monotonically increasing**. Several hits may arrive roughly at
the same time.

Implement the `HitCounter` class:

- `HitCounter()` initialises the object.
- `hit(timestamp)` records a hit that happened at `timestamp`.
- `getHits(timestamp)` returns the number of hits in the past 5 minutes from
  `timestamp` - that is, in the range `(timestamp - 300, timestamp]`.

---

### Example

```
HitCounter c = new HitCounter();
c.hit(1);            // hit at 1
c.hit(2);            // hit at 2
c.hit(3);            // hit at 3
c.getHits(4);        // 3   - window is (-296, 4]
c.hit(300);          // hit at 300
c.getHits(300);      // 4   - window is (0, 300], so 1, 2, 3 and 300
c.getHits(301);      // 3   - window is (1, 301], so the hit at 1 has just left
```

---

### Constraints

- `1 <= timestamp <= 2 * 10^9`
- All calls are made with a monotonically increasing `timestamp`
- At most `300` calls will be made to `hit` and `getHits`

**Follow-up:** what if the number of hits per second could be very large? Does your
design scale?

You have already built the sliding window in #933. This one changes two things, and
both of them matter: `getHits` asks a question **without** adding an event, and many
hits can share one timestamp. The follow-up is the real problem.

## Before you write anything

**1.** Put this next to #933 and list the differences. There are three: the window
length, the fact that a *query* is not also an *event*, and the fact that many hits
can land on the same second. Say which of the three changes the **data structure**
and which two only change the arithmetic.

**2.** The window is `(timestamp - 300, timestamp]` - open at the left, closed at
the right. Check that against the example: at `getHits(300)` the hit at `1` is
**counted**; at `getHits(301)` it is **gone**. Write the discard condition and then
test it on exactly those two calls. Getting this backwards is a one-character bug
that only shows up 300 seconds after you make it.

**3.** Route A keeps every hit in a queue. What is its memory, in terms of the
number of hits? Now read the follow-up: "the number of hits per second could be very
large" - say, a million hits a second on a busy web server. What is your queue
holding at that point, and how much of it is *the same number repeated*?

**4.** That repetition is the clue. You do not need to know *which* hits happened at
second `t`, only **how many**. So instead of one entry per hit, keep one entry per
**second**. How many seconds can be inside a 300-second window at once? That number
is a constant - which means your memory can be a constant too, no matter how many
hits arrive.

**5.** Now the trick that makes it `O(1)`: an array of exactly `300` slots, where the
hit at time `t` lives in slot `t % 300`. Two timestamps that land in the same slot
must be at least 300 apart - so at most one of them can be inside the window, and the
older one is always dead. What do you have to store *alongside* the count in each slot
to tell "300 hits at second 900" from "300 hits at second 600"? Answer that and you
have written route B.

**6.** How would you test it? `hit` returns nothing and `getHits` returns a number, so
a hit recorded into the wrong slot is invisible until a query happens to look at that
slot - possibly hundreds of calls later. What would you check after **every** call to
make it visible immediately?

## Two routes

**A - a queue of timestamps** *(write this first)*

```
self.q = deque()
```

`hit` appends. `getHits` pops from the front while the front is `<= timestamp - 300`,
then returns `len(self.q)`. Straight out of #933, and correct.

Costs: `O(1)` amortised per call, memory `O(hits in the window)`. Under LeetCode's
300-call limit that is at most 300 integers. Under the follow-up's million-hits-a-second
it is three hundred million integers, for a number that never exceeds that same count.

**B - a circular buffer of 300 slots** *(the follow-up)*

```
self.times  = [0] * 300      # which second slot i is currently holding
self.counts = [0] * 300      # how many hits that second saw
```

`hit(t)`: let `i = t % 300`. If `self.times[i] != t` then this slot is holding a
*stale* second - overwrite it (`times[i] = t`, `counts[i] = 1`); otherwise
`counts[i] += 1`. `getHits(t)`: sum `counts[i]` over the slots whose `times[i]` is
inside the window.

Both operations are `O(300)` worst case, which is `O(1)` - a constant that does not
grow. Memory is **600 integers, forever**, whether you take ten hits or ten billion.

> **The window has a fixed number of seconds in it, so give it a fixed number of
> slots.** That is the whole idea, and it is the same one behind #622's ring: when the
> thing you are tracking has a bounded size, allocate the bound once at construction
> and stop allocating. `t % 300` and "the next node in the ring" are the same
> operation, which #622 said out loud and this problem now makes you use.

In [ ]:
class HitCounter:

    def __init__(self):
        pass

    def hit(self, timestamp: int) -> None:
        pass

    def getHits(self, timestamp: int) -> int:
        pass

### The test harness

Question 6's answer. `hit` returns nothing, so a hit written into the wrong slot -
or a stale slot that was never cleared - is silent until some later `getHits` happens
to read it.

So `check` replays a call sequence against your class **and** against a brute-force
model that keeps every hit in a list. After **every** call, including every `hit`, it
asks your counter `getHits(now)` and compares to the model. That turns "a hit landed
in the wrong slot" into a failure at the call that caused it.

**What it deliberately does not do, and why.** An earlier version of this harness also
probed *ahead* - after a call at `t` it asked for `getHits(t + 301)` to test the window
boundary. That is an illegal test, and it is worth understanding why. Route A evicts
**lazily**, inside `getHits`: asking about `t + 301` throws away hits that are still
live at `t`. The next call then comes back at an earlier time and gets a wrong answer -
so a *correct* implementation fails. The problem guarantees timestamps only ever move
forwards, the class is entitled to rely on it, and a test that breaks that guarantee is
testing a class nobody asked you to write.

So the boundary cases from question 2 are driven as real, monotonically increasing call
sequences in the test cell instead - which is what a caller would actually do.

`stress` mixes dense bursts on one second with long jumps across the whole window. Run
this cell; don't edit it.

In [ ]:
import random

WINDOW = 300


def check(ops):
    '''Replay ("hit"|"getHits", t) against HitCounter and a keep-every-hit model.'''
    log = []
    try:
        hc = HitCounter()
    except Exception as e:
        return False, [f"   !! HitCounter() raised {type(e).__name__}: {e}"]

    hits = []                                   # the obviously-correct model

    def want(at):
        return sum(1 for h in hits if at - WINDOW < h <= at)

    for op, t in ops:
        call = f"{op}({t})"
        try:
            if op == "hit":
                hc.hit(t)
                hits.append(t)
                log.append(call)
            else:
                got = hc.getHits(t)
                log.append(f"{call} -> {got!r}   (want {want(t)}; window ({t - WINDOW}, {t}])")
                if got != want(t):
                    log.append(f"   !! {call} must return {want(t)}, got {got!r}")
                    return False, log
        except Exception as e:
            log.append(f"   !! {call} raised {type(e).__name__}: {e}")
            return False, log

        # After EVERY call, re-ask at the CURRENT time - never at a future one.
        # See the note above: probing the future would break the monotonic guarantee
        # the class is entitled to rely on.
        try:
            got = hc.getHits(t)
        except Exception as e:
            log.append(f"   !! after {call}, getHits({t}) raised {type(e).__name__}: {e}")
            return False, log
        if got != want(t):
            log.append(f"   !! after {call}, getHits({t}) is {got!r}, should be {want(t)}")
            log.append(f"      that window is ({t - WINDOW}, {t}]; "
                       f"hits so far: {hits[-8:]}{' ...' if len(hits) > 8 else ''}")
            return False, log

    return True, log


def stress(n, seed=0, burst=5, jump=200):
    '''Dense bursts on single seconds, plus long jumps across the window.'''
    random.seed(seed)
    ops, t = [], 1
    for _ in range(n):
        if random.random() < 0.7:
            for _ in range(random.randint(1, burst)):
                ops.append(("hit", t))
        ops.append(("getHits", t))
        t += random.randint(0, jump)
    return check(ops)


def report(name, ok, log, tail=6):
    print(f"{'OK  ' if ok else 'FAIL'} {name}")
    if not ok:
        for line in log[-tail:]:
            print(f"       {line}")

In [ ]:
# tests
CASES = [
    ("the LeetCode example",
     [("hit", 1), ("hit", 2), ("hit", 3), ("getHits", 4),
      ("hit", 300), ("getHits", 300), ("getHits", 301)]),

    ("no hits at all",                  [("getHits", 1), ("getHits", 10**9)]),
    ("one hit, then the window slides", [("hit", 1), ("getHits", 1), ("getHits", 300),
                                         ("getHits", 301), ("getHits", 5000)]),

    ("question 2: exactly 300 later is IN, 301 is OUT",
     [("hit", 100), ("getHits", 400), ("getHits", 401)]),

    ("question 3: many hits on one second",
     [("hit", 5)] * 50 + [("getHits", 5), ("getHits", 304), ("getHits", 305)]),

    ("question 5: two seconds exactly 300 apart share a slot",
     [("hit", 1), ("hit", 301), ("getHits", 301), ("getHits", 500), ("getHits", 601)]),

    ("question 5: a stale slot must not resurface",
     [("hit", 1), ("getHits", 1000), ("hit", 1000), ("getHits", 1000)]),

    ("question 2: the boundary, walked forwards one second at a time",
     [("hit", 10), ("hit", 10), ("hit", 11)]
     + [("getHits", t) for t in (11, 309, 310, 311, 312, 400)]),

    ("question 5: slots 300 apart, probed only forwards",
     [("hit", 7)] + [("getHits", t) for t in (7, 306, 307)]
     + [("hit", 307)] + [("getHits", t) for t in (307, 606, 607, 608)]),

    ("a hit on every second of a full window",
     [("hit", t) for t in range(1, 301)] + [("getHits", 300), ("getHits", 301),
                                            ("getHits", 600), ("getHits", 601)]),

    ("a long gap wipes the whole window",
     [("hit", 1), ("hit", 2), ("hit", 3), ("getHits", 3),
      ("getHits", 10**6), ("hit", 10**6), ("getHits", 10**6)]),

    ("the timestamp ceiling",
     [("hit", 2 * 10**9 - 1), ("getHits", 2 * 10**9 - 1), ("getHits", 2 * 10**9)]),
]

for name, ops in CASES:
    report(name, *check(ops))

for n, seed, burst, jump in [(20, 1, 3, 50), (60, 2, 5, 200), (100, 3, 20, 7), (200, 4, 2, 400)]:
    report(f"stress: {n} rounds (seed {seed}, bursts up to {burst}, jumps up to {jump}s)",
           *stress(n, seed, burst, jump))

print("\ntrace of the LeetCode example:")
for line in check([("hit", 1), ("hit", 2), ("hit", 3), ("getHits", 4),
                   ("hit", 300), ("getHits", 300), ("getHits", 301)])[1][:12]:
    print("  " + line)

## After it passes

- **Answer the follow-up with a number.** Take a million hits on one second, then run
  `getHits`. Print `len(self.q)` for route A and `len(self.times) + len(self.counts)`
  for route B. One is 1 000 000, the other is 600, and they return the same answer.
  That pair of numbers *is* the follow-up's answer - write it down rather than saying
  "it scales better".
- **Time them.** `timeit` a million `hit` calls in each. Route B does more arithmetic
  per call, so it may well be *slower* on this benchmark while being the only one that
  survives in production. Being able to say "slower per call, bounded in memory, and I
  chose it deliberately" is the point of building both.
- **Break the slot check.** Delete the `self.times[i] != t` test from route B and run
  the tests. Find which case fails - it will be "a stale slot must not resurface" - and
  say in one sentence why that single comparison is the entire difference between a
  circular buffer and a bug.
- **The invariant.** *Slot `i` holds a count for exactly one second, and that second is
  congruent to `i` mod 300.* Say which line of `hit` defends it.
- **Make it real.** Change the window from 300 seconds to a constructor argument, then
  ask what breaks if someone passes `86400` (a day). Then the question every metrics
  system answers: you cannot keep a slot per second for a year, so what do you do?
  (The words you want are *bucketing* and *downsampling* - one slot per second for an
  hour, one per minute for a day, one per hour for a year. That is Prometheus, Graphite
  and every dashboard you have used.)
- Siblings: **#933 Number of Recent Calls** (the same window, one event per call - do
  them back to back), #622 Design Circular Queue (the ring this borrows), #359 Logger
  Rate Limiter (a window per key), #1352 Product of the Last K Numbers.